In [2]:
from google.colab import drive
import os
from PIL import Image

print("🔗 Connecting to Google Drive...")
drive.mount('/content/drive')

zip_path = '/content/drive/MyDrive/SmartFarm_Dataset.zip'
extract_path = '/content/SmartFarm_Dataset'

print("📦 Extracting 16GB Dataset... (This takes 5-10 minutes)")
!unzip -q "{zip_path}" -d "{extract_path}"

print("🧹 Scanning for corrupted internet images to prevent crashes...")
corrupt_count = 0
for root, dirs, files in os.walk(extract_path):
    for file in files:
        if file.lower().endswith(('.png', '.jpg', '.jpeg')):
            file_path = os.path.join(root, file)
            try:
                img = Image.open(file_path)
                img.verify()
            except Exception:
                os.remove(file_path)
                corrupt_count += 1
print(f"✅ Removed {corrupt_count} corrupted files. Ready for training!")

🔗 Connecting to Google Drive...
Mounted at /content/drive
📦 Extracting 16GB Dataset... (This takes 5-10 minutes)
🧹 Scanning for corrupted internet images to prevent crashes...
✅ Removed 0 corrupted files. Ready for training!


In [6]:
import tensorflow as tf
import os
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping

extract_path = '/content/SmartFarm_Dataset'

# ==========================================
# FOOLPROOF PATH FINDER
# ==========================================
# This automatically hunts down the folder containing your 20 classes
for root, dirs, files in os.walk(extract_path):
    if len(dirs) > 5: # The master folder will have 20 subfolders in it
        extract_path = root
        print(f"🎯 Found true dataset folder at: {extract_path}")
        break

print("🧬 Setting up Advanced Data Augmentation...")
train_datagen = ImageDataGenerator(
    rescale=1./255, validation_split=0.2, rotation_range=25,
    width_shift_range=0.15, height_shift_range=0.15,
    zoom_range=0.2, horizontal_flip=True, brightness_range=[0.8, 1.2]
)
val_datagen = ImageDataGenerator(rescale=1./255, validation_split=0.2)

train_gen = train_datagen.flow_from_directory(
    extract_path, target_size=(224, 224), batch_size=32, class_mode='categorical', subset='training'
)
val_gen = val_datagen.flow_from_directory(
    extract_path, target_size=(224, 224), batch_size=32, class_mode='categorical', subset='validation'
)

# Failsafe check
if train_gen.num_classes != 20:
    print(f"⚠️ WARNING: Found {train_gen.num_classes} classes instead of 20! Check your zip file structure.")

print("🧠 Building Lightweight MobileNetV2 Architecture...")
base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base_model.trainable = False

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(256, activation='relu')(x)
x = Dropout(0.4)(x)
predictions = Dense(train_gen.num_classes, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=predictions)
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# ==========================================
# STAGE 1: Train Top Layers
# ==========================================
print("🔥 STAGE 1: Initializing AI Brain (8 Epochs) 🔥")
model.fit(train_gen, validation_data=val_gen, epochs=8)

# ==========================================
# STAGE 2: Deep Fine-Tuning
# ==========================================
print("🔥 STAGE 2: Deep Fine-Tuning for Maximum Accuracy 🔥")
base_model.trainable = True

fine_tune_at = len(base_model.layers) - 50
for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

model.compile(optimizer=tf.keras.optimizers.Adam(1e-5), loss='categorical_crossentropy', metrics=['accuracy'])
early_stop = EarlyStopping(monitor='val_accuracy', patience=4, restore_best_weights=True)

model.fit(train_gen, validation_data=val_gen, epochs=15, callbacks=[early_stop])

# ==========================================
# ACCURACY CHECK
# ==========================================
print("\n" + "="*50)
print("📊 RUNNING FINAL ACCURACY TEST ON UNSEEN DATA...")
loss, accuracy = model.evaluate(val_gen)
print(f"🏆 FINAL TRUE ACCURACY: {accuracy * 100:.2f}%")
print("="*50 + "\n")

# ==========================================
# EXPORT
# ==========================================
print("🗜️ Compressing Model for Flutter Offline Use...")
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()

with open('smartfarm_model.tflite', 'wb') as f:
    f.write(tflite_model)

labels = [k for k, v in sorted(train_gen.class_indices.items(), key=lambda item: item[1])]
with open('labels.txt', 'w') as f:
    for label in labels:
        f.write(f"{label}\n")

file_size = os.path.getsize('smartfarm_model.tflite') / (1024 * 1024)
print(f"🎉 COMPLETE! Model compressed to {file_size:.2f} MB.")

🎯 Found true dataset folder at: /content/SmartFarm_Dataset/Users/amrit/OneDrive/Desktop/SmartFarm_Dataset
🧬 Setting up Advanced Data Augmentation...
Found 18944 images belonging to 20 classes.
Found 4725 images belonging to 20 classes.
🧠 Building Lightweight MobileNetV2 Architecture...
🔥 STAGE 1: Initializing AI Brain (8 Epochs) 🔥
Epoch 1/8
592/592 ━━━━━━━━━━━━━━━━━━━━ 873s 1s/step - accuracy: 0.7649 - loss: 0.6904 - val_accuracy: 0.8358 - val_loss: 0.4671
Epoch 2/8
592/592 ━━━━━━━━━━━━━━━━━━━━ 829s 1s/step - accuracy: 0.8345 - loss: 0.4566 - val_accuracy: 0.8548 - val_loss: 0.4037
Epoch 3/8
592/592 ━━━━━━━━━━━━━━━━━━━━ 812s 1s/step - accuracy: 0.8534 - loss: 0.4015 - val_accuracy: 0.8656 - val_loss: 0.3761
Epoch 4/8
592/592 ━━━━━━━━━━━━━━━━━━━━ 817s 1s/step - accuracy: 0.8594 - loss: 0.3885 - val_accuracy: 0.8616 - val_loss: 0.3797
Epoch 5/8
592/592 ━━━━━━━━━━━━━━━━━━━━ 815s 1s/step - accuracy: 0.8632 - loss: 0.3706 - val_accuracy: 0.8686 - val_loss: 0.3578
Epoch 6/8
592/592 ━━━━━━━━━